In [17]:
import torch

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
%cd MyDrive

/content/drive/MyDrive


In [ ]:
%ls

In [5]:
from sequential_models import Linear
dataset_path = "fra.txt"

Remove excess text (attributions) and non-break space chars

In [6]:
def preprocess_text(text):
  text = text.split("CC-BY")[0].strip()
  text = text.replace("\u202f", " ").replace("\xa0", " ")
  no_space = lambda char, prev_char: char in ".,?!" and prev_char != " "
  out = [" " + char if i > 0 and no_space(char, text[i-1]) else char
         for i, char in enumerate(text.lower())]
  result = "".join(out)
  return(result)

Word-level tokenization

In [7]:
def tokenize_data(text, src, tgt):
  parts = text.split("\t")
  if len(parts) == 2:
    src.append([t for t in f"{parts[0]} <eos>".split(" ") if t])
    tgt.append([t for t in f"{parts[1]} <eos>".split(" ") if t])

In [57]:
x = torch.tensor([True, False, False, True]).type(torch.int32).sum(0)
x

tensor(2)

In [9]:
src = []
tgt = []

Preprocess and tokenize the data

In [10]:
with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

Pad or truncate the text to obtain a regular shape for computationally efficient operations

In [11]:
def pad_or_truncate(text, num_steps=10):
  if len(text) >= num_steps:
    text = text[:num_steps]
  else:
    text += ["<pad>"] * (num_steps - len(text))
  return text


Lookup table

In [12]:
from collections import Counter

def Vocab(src=src, min_freq=2):

  def flatten(src=src):
    return [item for text in src for item in text]

  flattened_seq = flatten()
  counter = Counter(flattened_seq)
  vocab = {"<unk>":0, "<bos>":1}
  i = 2

  for token, freq in counter.items():
    if freq > min_freq and token not in vocab.keys():
      vocab[token] = i
      i += 1
  return vocab

def encode(vocab, word):
  return vocab.get(word, vocab["<unk>"])



In [14]:
tgt[0]

['<bos>',
 'va',
 '!',
 '<eos>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>']

In [13]:
src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

In [15]:
lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]


In [18]:
src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

</bos/> is for input fed to the decoder. serves as a signal to the decoder to start producing an output. </eos/> is added to the label as a signal to the decoder to stop outputing tokens.

In [19]:
label = tgt_lookup[:, 1:]

label_valid_len = (label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
tgt_valid_len = (tgt_lookup != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)


In [ ]:
class Embedding():
  def __init__(self, vocab_size, feature_size):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.embedding = torch.randn((vocab_size, feature_size))

  def __call__(self):
    return self.embedding

  def parameters(self):
    return [self.embedding]

  def __repr__(self):
    return f"Embedding({self.embedding.shape})"